# 고군분투 AI - GPU 빠른 학습 (Colab / 로컬 GPU)1. 런타임 유형을 **GPU**로 바꿉니다 (Colab: 런타임 > 런타임 유형 변경 > T4 GPU 등).2. `gogun_ai_project.zip`을 왼쪽 파일 패널에 업로드합니다.3. 아래 셀을 위에서부터 실행합니다. 학습이 끝나면 `weights_*.json`을 내려받아 웹 페이지의 **TRAIN > IMPORT** 로 불러오면 됩니다.환경(게임 시뮬레이터)은 C 코드가 **모든 CPU 코어**(OpenMP)로 돌리고, 신경망 학습은 GPU가 담당합니다.

In [ ]:
!unzip -q -o gogun_ai_project.zip%cd game!./build.sh!nproc!nvidia-smi -L

## 1. 생존 전용 모델 (관측 64) - 처음부터Colab T4 + 4코어 기준 수 분이면 전체 게임을 거의 항상 클리어합니다. (CPU 1코어 numpy 백엔드로도 3분에 99 % 클리어를 확인했습니다.)

In [ ]:
!python train_gpu.py --device cuda --envs 8192 --T 128 --mb 32768 --hidden 128 --minutes 8 --eval_every 40 --out ck_surv

## 2. 코인 인식 모델 (관측 84 = 64 + 코인 20, 보상에 코인 점수 포함)`--coin_w` 는 코인 1점당 보상입니다. 클수록 코인을 적극적으로 노리지만 사망 위험도 늘어납니다 (0.0005 ~ 0.002 권장).

In [ ]:
!python train_gpu.py --device cuda --obs 2 --coin_w 0.001 --init ck_surv_ema.npz --lr 1e-4 --ent 0.003 --p_trans 0.2 --minutes 10 --eval_every 40 --out ck_coin

## 3. 대결 점수(거리 + 코인)로 평가하고 웹용 JSON 내보내기

In [ ]:
!python eval_battle.py ck_surv_ema.npz ck_coin_ema.npz --games 200!python export_ck.py ck_coin_ema.npz weights_coin.json!python export_ck.py ck_surv_ema.npz weights_surv.json

In [ ]:
from google.colab import filesfiles.download('weights_coin.json'); files.download('weights_surv.json')